# LangGraph Fundamentals — Stateful Workflows

This notebook walks through every core LangGraph concept, from a minimal 2-node graph to human-in-the-loop workflows.

**What you'll learn:**
1. StateGraph — the basic building block
2. State with TypedDict — shared data across nodes
3. LLM nodes — calling models inside a graph
4. Web search nodes — integrating Tavily
5. Conditional edges — dynamic routing
6. Loops — revision/retry patterns
7. Checkpointing — persisting state across invocations
8. Human-in-the-loop — pausing for approval

### LangGraph vs LangChain chains

| | LangChain Chain (`\|`) | LangGraph |
|---|---|---|
| Data flow | Output of step N → input of step N+1 | All nodes read/write **shared state** |
| Control flow | Always linear | Conditional edges, loops, branches |
| Persistence | None | Built-in checkpointing |
| Human-in-the-loop | Not supported | `interrupt()` / `Command(resume=...)` |

## Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)
print(f"Loaded .env from: {env_path}")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

---
## 1. Minimal StateGraph — Two Nodes in Sequence

A `StateGraph` has three ingredients:
1. **State** — a `TypedDict` that all nodes share
2. **Nodes** — functions that read state and return partial updates
3. **Edges** — connections between nodes

Special constants:
- `START` — the entry point (not a real node)
- `END` — the exit point (not a real node)

**Key methods on `StateGraph`:**
- `.add_node(name, function)` — register a node
- `.add_edge(from, to)` — connect two nodes
- `.compile()` — build the runnable graph

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class GreetingState(TypedDict):
    name: str
    greeting: str

def greet(state: GreetingState) -> dict:
    return {"greeting": f"Hello, {state['name']}!"}

def shout(state: GreetingState) -> dict:
    return {"greeting": state["greeting"].upper()}

builder = StateGraph(GreetingState)
builder.add_node("greet", greet)
builder.add_node("shout", shout)
builder.add_edge(START, "greet")
builder.add_edge("greet", "shout")
builder.add_edge("shout", END)

graph = builder.compile()

# Visualize
print("Mermaid diagram:")
print(graph.get_graph().draw_mermaid())

In [ ]:
result = graph.invoke({"name": "Workshop", "greeting": ""})
print(f"Result: {result}")

### How state updates work

Each node returns a **partial dict** — only the fields it wants to change.

```python
def greet(state) -> dict:
    return {"greeting": "Hello!"}  # only updates 'greeting', leaves 'name' alone
```

The graph merges this into the state automatically. Fields not returned stay unchanged.

---
## 2. LLM Nodes — Calling Models Inside a Graph

A node is just a function. If that function calls an LLM and writes the result to state, you've got an LLM node.

In [ ]:
class JokeState(TypedDict):
    topic: str
    joke: str
    rating: str

def write_joke(state: JokeState) -> dict:
    response = llm.invoke([
        ("system", "Write a short one-liner joke. Just the joke, nothing else."),
        ("user", f"Topic: {state['topic']}"),
    ])
    return {"joke": response.content}

def rate_joke(state: JokeState) -> dict:
    response = llm.invoke([
        ("system", "Rate this joke from 1-10. Reply with just the number and a one-word reason."),
        ("user", state["joke"]),
    ])
    return {"rating": response.content}

builder = StateGraph(JokeState)
builder.add_node("comedian", write_joke)
builder.add_node("critic", rate_joke)
builder.add_edge(START, "comedian")
builder.add_edge("comedian", "critic")
builder.add_edge("critic", END)

graph = builder.compile()
result = graph.invoke({"topic": "Python programming", "joke": "", "rating": ""})

print(f"Topic:  {result['topic']}")
print(f"Joke:   {result['joke']}")
print(f"Rating: {result['rating']}")

Notice how the `critic` node reads the `joke` field that `comedian` wrote — that's the shared state in action. In an LCEL chain, the critic would only see the comedian's raw output, not the original topic.

---
## 3. Web Search Nodes — Integrating Tavily

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_results=2)

class ResearchState(TypedDict):
    query: str
    raw_results: str
    summary: str

def search(state: ResearchState) -> dict:
    response = tavily.invoke(state["query"])
    results = response["results"]
    text = "\n".join(f"- [{r['title']}] {r['content'][:200]}" for r in results)
    return {"raw_results": text}

def summarize(state: ResearchState) -> dict:
    response = llm.invoke([
        ("system", "Summarize these search results in 2-3 bullet points."),
        ("user", state["raw_results"]),
    ])
    return {"summary": response.content}

builder = StateGraph(ResearchState)
builder.add_node("search", search)
builder.add_node("summarize", summarize)
builder.add_edge(START, "search")
builder.add_edge("search", "summarize")
builder.add_edge("summarize", END)

graph = builder.compile()
result = graph.invoke({"query": "LangGraph vs LangChain 2026", "raw_results": "", "summary": ""})

print("Summary:")
print(result["summary"])

---
## 4. Conditional Edges — Dynamic Routing

Instead of always going A → B, a **conditional edge** calls a function that returns the name of the next node.

**Key method:**
```python
builder.add_conditional_edges(
    "source_node",
    routing_function,       # returns a string like "approve" or "reject"
    {"approve": "approve", "reject": "reject"}  # maps return values to node names
)
```

In [ ]:
class ReviewState(TypedDict):
    text: str
    word_count: int
    verdict: str

def count_words(state: ReviewState) -> dict:
    return {"word_count": len(state["text"].split())}

def approve(state: ReviewState) -> dict:
    return {"verdict": f"Approved ({state['word_count']} words)"}

def reject(state: ReviewState) -> dict:
    return {"verdict": f"Too short — only {state['word_count']} words, need 5+"}

def route(state: ReviewState) -> str:
    return "approve" if state["word_count"] >= 5 else "reject"

builder = StateGraph(ReviewState)
builder.add_node("count", count_words)
builder.add_node("approve", approve)
builder.add_node("reject", reject)

builder.add_edge(START, "count")
builder.add_conditional_edges("count", route, {
    "approve": "approve",
    "reject": "reject",
})
builder.add_edge("approve", END)
builder.add_edge("reject", END)

graph = builder.compile()
print("Mermaid diagram:")
print(graph.get_graph().draw_mermaid())

In [ ]:
# Takes the "approve" path
r1 = graph.invoke({"text": "LangGraph makes it easy to build stateful workflows", "word_count": 0, "verdict": ""})
print(f"Long text:  {r1['verdict']}")

# Takes the "reject" path
r2 = graph.invoke({"text": "Hello world", "word_count": 0, "verdict": ""})
print(f"Short text: {r2['verdict']}")

---
## 5. Loops — Revision/Retry Patterns

Conditional edges can point **backwards**, creating a loop. This is the key pattern for:
- Write → review → revise → review → ... → approve
- Generate → validate → regenerate → ... → valid

Always include a **max iterations** guard to prevent infinite loops.

In [ ]:
class PasswordState(TypedDict):
    password: str
    is_strong: bool
    attempts: int
    result: str

import random
import string

def generate_password(state: PasswordState) -> dict:
    length = 6 + state["attempts"] * 4  # gets longer each attempt
    chars = string.ascii_letters + string.digits
    if state["attempts"] > 0:
        chars += "!@#$%"  # add special chars after first attempt
    pw = "".join(random.choice(chars) for _ in range(length))
    print(f"  Attempt {state['attempts'] + 1}: generated '{pw}' ({len(pw)} chars)")
    return {"password": pw, "attempts": state["attempts"] + 1}

def check_strength(state: PasswordState) -> dict:
    pw = state["password"]
    strong = (
        len(pw) >= 10
        and any(c.isupper() for c in pw)
        and any(c.isdigit() for c in pw)
        and any(c in "!@#$%" for c in pw)
    )
    return {"is_strong": strong}

def finalize(state: PasswordState) -> dict:
    return {"result": f"Strong password found after {state['attempts']} attempts: {state['password']}"}

def should_retry(state: PasswordState) -> str:
    if state["is_strong"] or state["attempts"] >= 5:
        return "finalize"
    return "generate"

builder = StateGraph(PasswordState)
builder.add_node("generate", generate_password)
builder.add_node("check", check_strength)
builder.add_node("finalize", finalize)

builder.add_edge(START, "generate")
builder.add_edge("generate", "check")
builder.add_conditional_edges("check", should_retry, {
    "generate": "generate",  # loop back!
    "finalize": "finalize",
})
builder.add_edge("finalize", END)

graph = builder.compile()
result = graph.invoke({"password": "", "is_strong": False, "attempts": 0, "result": ""})
print(f"\n{result['result']}")

---
## 6. Checkpointing — Persisting State

A **checkpointer** saves the graph state at each step. This enables:
- Resume after a crash
- Multi-turn conversations (same `thread_id` = same state)
- Human-in-the-loop (pause, wait for input, resume)

**Built-in checkpointers:**

| Checkpointer | Use case | Package |
|---|---|---|
| `MemorySaver` | Development / testing | `langgraph` (built-in) |
| `SqliteSaver` | Local persistence | `langgraph` (built-in) |
| `PostgresSaver` | Production | `langgraph-checkpoint-postgres` |

**Required:** pass `thread_id` in config so the checkpointer knows which conversation you're in.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

class CounterState(TypedDict):
    count: int

def increment(state: CounterState) -> dict:
    return {"count": state["count"] + 1}

builder = StateGraph(CounterState)
builder.add_node("increment", increment)
builder.add_edge(START, "increment")
builder.add_edge("increment", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Same thread_id = state persists across calls
config = {"configurable": {"thread_id": "demo-counter"}}

r1 = graph.invoke({"count": 0}, config)
print(f"Call 1: count = {r1['count']}")

r2 = graph.invoke({"count": r1["count"]}, config)
print(f"Call 2: count = {r2['count']}")

r3 = graph.invoke({"count": r2["count"]}, config)
print(f"Call 3: count = {r3['count']}")

# Different thread_id = fresh state
other_config = {"configurable": {"thread_id": "other-thread"}}
r4 = graph.invoke({"count": 0}, other_config)
print(f"New thread: count = {r4['count']}")

---
## 7. Human-in-the-Loop — Pausing for Approval

The `interrupt()` function **pauses** the graph and returns control to the caller. The caller can inspect the state, get human input, and **resume** with `Command(resume=value)`.

**Requirements:**
- Must compile with a checkpointer
- Must pass `thread_id` in config
- Interrupt payload must be JSON-serializable

In [ ]:
from langgraph.types import Command, interrupt

class ApprovalState(TypedDict):
    proposal: str
    approved: bool
    result: str

def draft_proposal(state: ApprovalState) -> dict:
    response = llm.invoke([
        ("system", "Write a one-sentence project proposal."),
        ("user", state["proposal"]),
    ])
    return {"proposal": response.content}

def human_review(state: ApprovalState) -> dict:
    # This pauses the graph and returns the payload to the caller
    decision = interrupt({
        "proposal": state["proposal"],
        "question": "Do you approve this proposal? (yes/no)",
    })
    return {"approved": decision.lower() == "yes"}

def finalize(state: ApprovalState) -> dict:
    status = "APPROVED" if state["approved"] else "REJECTED"
    return {"result": f"{status}: {state['proposal']}"}

builder = StateGraph(ApprovalState)
builder.add_node("draft", draft_proposal)
builder.add_node("review", human_review)
builder.add_node("finalize", finalize)

builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", "finalize")
builder.add_edge("finalize", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-demo"}}

# First call — runs until it hits interrupt()
result = graph.invoke(
    {"proposal": "Build an AI-powered code review tool", "approved": False, "result": ""},
    config,
)

# Check if the graph paused
snapshot = graph.get_state(config)
print(f"Graph paused at: {snapshot.next}")
print(f"Proposal: {result['proposal']}")

In [ ]:
# Resume with human decision
print("Simulating human approval: 'yes'")
result = graph.invoke(Command(resume="yes"), config)
print(f"Result: {result['result']}")

---
## Summary — LangGraph Building Blocks

```
StateGraph          →  Define state (TypedDict) + nodes (functions) + edges
add_edge            →  Fixed connection: always go from A to B
add_conditional_edges →  Dynamic routing: a function decides the next node
Loops               →  Conditional edges pointing backwards (with max iterations)
MemorySaver         →  Persist state across calls using thread_id
interrupt()         →  Pause the graph, wait for external input
Command(resume=...) →  Resume the graph with a value
```

**Key insight:** LangGraph gives you explicit control over the workflow, unlike an agent where the LLM decides everything.

**Next up:** Open `starter.py` and build a Blog Writer workflow with planning, research, writing, and review — with revision loops and human approval.